[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MrEGAMERZ/netguard-ai/blob/main/NetGuard_AI_Explained.ipynb)

# NetGuard AI — Explained Version
## Network Intrusion Detection System (Line-by-Line Walkthrough)

**What this notebook does in simple words:**

Imagine the internet is a road. Every time your computer sends or receives data, a "car" (network packet) drives on that road. Most cars are normal — they carry emails, YouTube videos, Zoom calls. But some cars are attackers trying to break into your network.

This notebook teaches a computer to look at measurements of these cars (how fast they go, how big they are, how often they come) and decide: **Is this safe traffic or an attack?**

We do this using two different methods:
1. **Random Forest** — Like asking 100 different security guards to vote
2. **Neural Network** — Like training an artificial brain to recognise attack patterns

**Dataset:** CIC-IDS2017 — a research dataset from the Canadian Institute for Cybersecurity. It has real recorded network traffic with both normal and attack examples labelled.

> **To use GPU (makes the neural network faster):**
> Runtime → Change runtime type → T4 GPU


## Step 1 — Install the Extra Library

Python comes with many built-in tools, but we need one extra package called `datasets`.
It is made by Hugging Face (a company that hosts machine learning data).
This library lets us stream network traffic data from the internet without
downloading the entire 50 GB file all at once.

The `!` at the start of the line means: run this in the terminal, not as Python code.
The `-q` flag means "quiet" — don't print every small status message.


In [ ]:
# The ! tells Colab: run this as a system (terminal) command, not Python
# pip is Python's package manager — it downloads and installs libraries
# install = download and set up the library
# -q = quiet mode (suppress most of the output messages)
# datasets = the name of the library we want from Hugging Face
!pip install -q datasets

# This just prints a message so we know the cell finished running
print("Library installed. Ready to go.")


## Step 2 — Load All the Tools We Will Use

Before cooking, you get out all your ingredients and utensils.
Here we do the same — we tell Python which tools (libraries) to load.

Each `import` line loads a toolkit. Think of each one as a specialised toolbox:

| Library | What it does |
|---|---|
| `numpy` | Fast math on large lists of numbers |
| `pandas` | Load tables of data (like Excel, but in Python) |
| `matplotlib` | Draw basic charts and graphs |
| `seaborn` | Draw prettier, more detailed charts |
| `datasets` | Download our network traffic data from Hugging Face |
| `sklearn` | Machine learning tools — Random Forest, train/test split, scoring |
| `tensorflow / keras` | Build and train neural networks |
| `warnings` | Control warning messages so they don't clutter our output |


In [ ]:
# numpy: fast number crunching on arrays (lists of numbers)
# We nickname it 'np' so we can type np.something instead of numpy.something
import numpy as np

# pandas: works with tables of data (rows and columns), like a spreadsheet
# We nickname it 'pd'
import pandas as pd

# matplotlib.pyplot: draws graphs and charts
# We nickname it 'plt'
import matplotlib.pyplot as plt

# seaborn: a nicer-looking chart library built on top of matplotlib
# We nickname it 'sns'
import seaborn as sns

# From the Hugging Face 'datasets' library, import the load_dataset function
# load_dataset() is what we use to stream the network traffic data
from datasets import load_dataset

# From scikit-learn (sklearn), import specific tools:
# train_test_split  — splits data into training and testing portions
# StandardScaler    — normalises numbers so they are on a similar scale
# RandomForestClassifier — the Random Forest model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# More scikit-learn tools — these measure how well our model performs
# accuracy_score   — what % of predictions were correct overall
# precision_score  — of all things we called "Attack", how many were really attacks?
# recall_score     — of all real attacks, how many did we catch?
# f1_score         — a single number that balances precision and recall
# confusion_matrix — a 2x2 grid showing correct vs wrong predictions
# classification_report — prints all metrics at once in a readable table
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# tensorflow: Google's library for building and running neural networks
import tensorflow as tf

# keras: a simpler interface on top of tensorflow
# keras lets us build neural networks layer by layer, like stacking Lego blocks
from tensorflow import keras

# layers: the building blocks of a neural network (Dense, Dropout, etc.)
from tensorflow.keras import layers

# warnings: some libraries print confusing warning messages; this suppresses them
import warnings
# filterwarnings("ignore") = hide all warnings so the output is clean
warnings.filterwarnings("ignore")

# RANDOM_STATE = 42 is just a number we use to make random operations repeatable
# Example: if we shuffle data, using the same seed gives the same shuffle every time
# 42 is a programmer tradition — it has no special meaning mathematically
RANDOM_STATE = 42

# Fix numpy's random number generator with our seed so results are reproducible
np.random.seed(RANDOM_STATE)

# Fix TensorFlow's random number generator for the same reason
tf.random.set_seed(RANDOM_STATE)

# Check if Colab gave us a GPU (graphics card) for faster computation
# If tf.config.list_physical_devices("GPU") returns something, we have a GPU
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"

# Print which device we are using so we know
print(f"All tools loaded. This notebook is running on: {device}")
print("If it says CPU and you want faster training, go to Runtime -> Change runtime type -> T4 GPU")


## Step 3 — Download the Network Traffic Data

### What is CIC-IDS2017?

The **Canadian Institute for Cybersecurity** recorded real network traffic in a lab in 2017.
They let attackers run real cyberattacks and recorded everything — the normal traffic AND the attacks.
Then they labelled each record: "Benign" (safe) or the type of attack (DDoS, PortScan, etc.).

The result is a research dataset that universities and companies use to test intrusion detection systems.

### Why these two specific files?

The dataset is split by day of the week:
- **Monday** = only normal traffic (no attacks at all — useless for us)
- **Friday afternoon** = contains two attack types: DDoS and PortScan ✅

We pick Friday's files so we definitely have both safe and attack examples.

### What is DDoS?
DDoS stands for **Distributed Denial of Service**. Imagine hundreds of computers all calling your phone at the same time so no real caller can get through. The goal is to crash or slow down a server.

### What is PortScan?
A port scan is like a burglar walking down a street trying every door handle to see which ones are unlocked. An attacker scans all the "ports" (communication channels) of a computer to find open ones to exploit.

### Why collect by class instead of just taking the first 200,000 records?

If we just grab the first 200,000 records from the dataset, we might get only Benign records (because the file starts with normal traffic). We need BOTH classes. So we deliberately collect:
- **70,000 Benign** records (35,000 from each file)
- **15,000 DDoS** attack records
- **15,000 PortScan** attack records

Total: **100,000 records** — balanced enough to train a fair model.


In [ ]:
# Tell the user what we are about to do
print("Loading balanced network traffic sample from CIC-IDS2017...")
print("This will take 2-5 minutes. The data is streaming from the internet.")
print()

# ── Load the DDoS file ────────────────────────────────────────────────────
# load_dataset() connects to the Hugging Face dataset repository
# "muzom/CIC-IDS2017" = the dataset name on Hugging Face (owner/dataset)
# data_files = we tell it EXACTLY which file inside the dataset to open
#   (the Friday afternoon DDoS capture file)
# split="train" = use the training portion (Hugging Face datasets have splits)
# streaming=True = do NOT download the whole file first; read record by record
#   This saves memory — we only keep what we need
ds_ddos = load_dataset(
    "muzom/CIC-IDS2017",
    data_files=["Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"],
    split="train",
    streaming=True,
)

# Same thing, but for the PortScan capture file
ds_port = load_dataset(
    "muzom/CIC-IDS2017",
    data_files=["Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"],
    split="train",
    streaming=True,
)

# Create three empty lists — we will fill them as we read the data
benign = []    # will hold normal (safe) traffic records
ddos = []      # will hold DDoS attack records
portscan = []  # will hold PortScan attack records

# ── Collect records from the DDoS file ───────────────────────────────────
# ds_ddos is a stream — we loop through it one record at a time
# Each 'item' is a dictionary like {"Flow Duration": 100, "Label": "BENIGN", ...}
for item in ds_ddos:
    # Get the label — in some rows the column is called " Label" (with a space)
    # .get() checks both names; if neither exists, use empty string ""
    # .strip() removes leading/trailing spaces from the label text
    label = item.get(" Label", item.get("Label", "")).strip()

    # If the label says BENIGN (we use .upper() so "benign"/"BENIGN" both match)
    # AND we still need more benign records (less than 35,000)
    if label.upper() == "BENIGN" and len(benign) < 35000:
        benign.append(item)   # add this record to our benign list

    # If the label is NOT benign (so it is some kind of attack)
    # AND we still need more DDoS records (less than 15,000)
    elif label.upper() != "BENIGN" and len(ddos) < 15000:
        ddos.append(item)     # add this record to our DDoS list

    # Once we have enough of BOTH types from this file, stop reading
    # (No point reading millions more records once our buckets are full)
    if len(benign) >= 35000 and len(ddos) >= 15000:
        break

print(f"From DDoS file: {len(benign)} Benign, {len(ddos)} DDoS records collected")

# ── Collect records from the PortScan file ───────────────────────────────
# Same logic as above, but we continue filling 'benign' up to 70,000
# (another 35,000 from this second file)
for item in ds_port:
    label = item.get(" Label", item.get("Label", "")).strip()

    # We want 70,000 benign total (35k from DDoS file + 35k from this file)
    if label.upper() == "BENIGN" and len(benign) < 70000:
        benign.append(item)

    # We want 15,000 PortScan records
    elif label.upper() != "BENIGN" and len(portscan) < 15000:
        portscan.append(item)

    # Stop once both buckets for this file are full
    if len(benign) >= 70000 and len(portscan) >= 15000:
        break

print(f"From PortScan file: {len(benign)} Benign total, {len(portscan)} PortScan records collected")
print()

# ── Combine all records into one table ───────────────────────────────────
# benign + ddos + portscan = concatenate the three lists into one big list
# pd.DataFrame() converts a list of dictionaries into a table (rows + columns)
# .sample(frac=1.0, ...) = shuffle ALL rows randomly (frac=1.0 means 100% of them)
#   random_state=RANDOM_STATE = use our fixed seed so the shuffle is reproducible
# .reset_index(drop=True) = after shuffling, reset row numbers from 0
df = pd.DataFrame(benign + ddos + portscan).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

# Strip leading/trailing spaces from all column names
# (Some column names in this dataset have accidental spaces like " Label")
df.columns = df.columns.str.strip()

# Print a summary of what we loaded
print(f"Dataset ready: {df.shape[0]:,} total records, {df.shape[1]} columns")
print()
# .value_counts() counts how many rows have each unique label value
print("Records per label:")
print(df["Label"].value_counts())

# Show the first 3 rows as a preview table
df.head(3)


## Step 4 — Explore and Understand the Data (EDA)

Before we train any model, we look at the data carefully.
This is called **Exploratory Data Analysis (EDA)**.

Think of it like checking your ingredients before you start cooking:
- How many columns (features) do we have?
- Are any values missing?
- How many Benign vs Attack records are there?

This step does not change anything — it just prints information so we understand what we are working with.


In [ ]:
# df.columns is a list of all column names (feature names)
# len() counts how many there are
# .tolist() converts it to a plain Python list
print(f"Total number of columns (features): {len(df.columns)}")
print()

# Print the first 20 column names so we can see what measurements we have
# Each column represents one measurement of a network connection
# e.g. "Flow Duration" = how long the connection lasted in microseconds
print("First 20 column names:")
print(df.columns.tolist()[:20])
print("... (and more)")


In [ ]:
# df.info() prints a summary of the whole table:
# - How many rows total
# - Each column name, how many non-null values it has, and its data type
# 'object' type = text/string columns
# 'float64' type = decimal number columns
# 'int64' type = whole number columns
print("Dataset summary (types and non-null counts):")
df.info()


In [ ]:
# Check for missing values in each column
# df.isnull() creates a True/False table (True = that cell is empty/missing)
# .sum() counts how many Trues (missing values) are in each column
missing = df.isnull().sum()

print("Missing values per column (showing only columns that have missing data):")
# Show only columns where missing count is more than 0
# .sort_values(ascending=False) = show the worst columns first
print(missing[missing > 0].sort_values(ascending=False).head(10))
print()

# Show the class distribution — how many records of each label do we have?
print("How many records per label (before we convert to binary):")
# .value_counts() counts occurrences of each unique value in the "Label" column
print(df["Label"].value_counts())


In [ ]:
# Draw a bar chart showing how many records we have for each original label
# This helps us visually see the class balance

# .value_counts() gives us {label: count} in descending order
label_counts = df["Label"].value_counts()

# Create a figure (canvas) that is 12 inches wide and 5 inches tall
plt.figure(figsize=(12, 5))

# Draw a vertical bar chart
# label_counts.index = the category labels (Benign, DDoS, PortScan)
# label_counts.values = the count for each
# color = bar fill color (a nice blue)
# edgecolor = bar border color
# linewidth = how thick the border line is
plt.bar(label_counts.index, label_counts.values, color="#4a90d9", edgecolor="black", linewidth=0.4)

# Rotate the x-axis labels so they don't overlap
# ha="right" = right-align the text
# fontsize=9 = make the text a bit smaller
plt.xticks(rotation=45, ha="right", fontsize=9)

# Label the y-axis
plt.ylabel("Number of Records")

# Title for the chart
plt.title("Original Traffic Labels (100,000 record sample)")

# tight_layout() adjusts spacing so nothing gets cut off
plt.tight_layout()

# Actually display the chart
plt.show()


## Step 5 — Clean and Prepare the Data

Raw data is messy. We fix four things before training:

### 1. Convert labels to numbers
Computers work with numbers, not words. We convert:
- "BENIGN" → `0` (safe)
- Anything else (DDoS, PortScan) → `1` (attack)

This is called **binary classification** — only two possible answers.

### 2. Fix infinite and missing values
Some network flow calculations produce infinity (e.g. dividing by zero when a connection had zero duration). We replace these with the median value of that column.

### 3. Select useful features
Not all 80+ columns are useful. We pick 24 specific measurements that are known to be informative for detecting attacks.

### 4. Why do we keep things simple?
Adding more features does not always make the model better. It can even make it worse (called the "curse of dimensionality"). 24 well-chosen features is enough.


In [ ]:
# ── Step 5a: Convert text labels to numbers ──────────────────────────────

# Create a new column called "binary_label"
# df["Label"].apply(function) = run a function on every row's Label value
# lambda x: ... = a small one-line function where x is the current label value
# If the label (stripped and uppercased) equals "BENIGN" → return 0
# Otherwise → return 1  (it is some kind of attack)
df["binary_label"] = df["Label"].apply(
    lambda x: 0 if str(x).strip().upper() == "BENIGN" else 1
)

# Count how many benign (0) and attack (1) records we have
# (df["binary_label"] == 0).sum() counts all rows where binary_label is 0
benign_cnt = (df["binary_label"] == 0).sum()
attack_cnt = (df["binary_label"] == 1).sum()
total = len(df)   # total number of rows

# Print the counts and percentages
print(f"Benign records: {benign_cnt:,} ({benign_cnt/total*100:.1f}% of total)")
print(f"Attack records: {attack_cnt:,} ({attack_cnt/total*100:.1f}% of total)")

# Draw a bar chart of the binary distribution
plt.figure(figsize=(5, 4))
# Two bars: Benign and Attack
# Colors: green for Benign, red for Attack (intuitive traffic light colours)
plt.bar(["Benign (0)", "Attack (1)"], [benign_cnt, attack_cnt],
        color=["#2ca02c", "#d62728"], edgecolor="black", linewidth=0.5)
plt.ylabel("Count")
plt.title("Binary Label Distribution")

# Add the actual count number above each bar
# enumerate() gives us (index, value) pairs: (0, benign_cnt), (1, attack_cnt)
for i, v in enumerate([benign_cnt, attack_cnt]):
    # plt.text(x, y, text) draws text at position (x, y) on the chart
    # v + 2000 puts the text just above the top of the bar
    plt.text(i, v + 2000, f"{v:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# ── Step 5b: Find numeric columns and fix infinite / missing values ───────

# .select_dtypes(include=[np.number]) returns only columns with number types
# .columns gives the column names, .tolist() makes it a plain Python list
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove "binary_label" from this list — it is our answer (target), not a feature
if "binary_label" in numeric_cols:
    numeric_cols.remove("binary_label")

# Replace infinite values with NaN (Not a Number — Python's way of saying "missing")
# np.inf = positive infinity, -np.inf = negative infinity
# Some flow metrics can produce infinity if the connection duration was 0
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Fill each NaN with the median of that column
# .median() gives the middle value — more robust than the average for skewed data
# .fillna() replaces NaN values with the given values
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Check which columns are now completely clean (no missing values)
# This is a list of column names where the missing count is 0
valid_cols = [c for c in numeric_cols if df[c].isnull().sum() == 0]

print(f"Total numeric columns: {len(numeric_cols)}")
print(f"Columns with no missing values after cleaning: {len(valid_cols)}")


In [ ]:
# ── Step 5c: Select the 24 most useful features ───────────────────────────

# This is our hand-picked list of 24 network flow measurements
# These are well-known features used in network intrusion detection research
# Each one captures something different about how a connection behaves
preferred_features = [
    "Flow Duration",             # How long the connection lasted (microseconds)
    "Total Fwd Packets",         # How many packets were sent TO the destination
    "Total Backward Packets",    # How many packets came BACK from the destination
    "Total Length of Fwd Packets",  # Total bytes sent forward
    "Total Length of Bwd Packets",  # Total bytes received back
    "Flow Bytes/s",              # Average data transfer rate (bytes per second)
    "Flow Packets/s",            # Average packet rate (packets per second)
    "Flow IAT Mean",             # Average time gap between packets (IAT = Inter-Arrival Time)
    "Flow IAT Std",              # How consistent those gaps are (standard deviation)
    "Fwd Packet Length Mean",    # Average size of packets sent forward
    "Bwd Packet Length Mean",    # Average size of packets coming back
    "Packet Length Mean",        # Average packet size overall
    "Packet Length Std",         # How much packet sizes vary
    "Average Packet Size",       # Another way to measure average packet size
    "Fwd Packets/s",             # How fast packets were sent forward
    "Bwd Packets/s",             # How fast packets came back
    "Init_Win_bytes_forward",    # Initial TCP window size (how much data can flow at once)
    "Init_Win_bytes_backward",   # Same for the reverse direction
    "ACK Flag Count",            # How many packets had the ACK flag (acknowledgement)
    "SYN Flag Count",            # How many packets had the SYN flag (connection start)
    "PSH Flag Count",            # How many packets had the PSH flag (push data immediately)
    "RST Flag Count",            # How many packets had the RST flag (connection reset/error)
    "Active Mean",               # Average time the connection was actively sending data
    "Idle Mean",                 # Average time the connection was idle (waiting)
]

# Keep only features that actually exist in our dataset AND have no missing values
# (Some columns might have been named slightly differently or dropped during cleaning)
selected_features = [f for f in preferred_features if f in valid_cols]

# Safety net: if we somehow lost most features, just use all valid numeric columns
if len(selected_features) < 10:
    selected_features = valid_cols
    print("Warning: fell back to all valid columns")

print(f"We will train our models using {len(selected_features)} features:")
# Print each feature name on its own line for readability
for i, feat in enumerate(selected_features, 1):
    print(f"  {i:2d}. {feat}")


## Step 6 — Split the Data and Normalise the Numbers

### Train / Test Split

We split our 100,000 records into two groups:
- **80,000 for training** — the model studies these to learn patterns
- **20,000 for testing** — the model has NEVER seen these; we use them for the final exam

The test set is locked away during training. We only open it at the very end to get an honest score. If we tested on training data, it would be like a student memorising the answers — the score would be fake.

`stratify=y` ensures that both groups have the same ratio of Benign to Attack records. Without this, we might accidentally put all the attacks in one group.

### StandardScaler (Normalisation)

Different features have wildly different scales:
- "Flow Duration" might be 500,000 (microseconds)
- "SYN Flag Count" might be 1

If we feed these raw numbers to a neural network, the large ones dominate and the model ignores the small ones. StandardScaler transforms every feature so they all have:
- Mean = 0
- Standard deviation = 1

This makes training much more stable and fair.

> **Important:** We fit the scaler ONLY on training data. We then apply the same transformation to the test data. This prevents "data leakage" — the model must not know anything about the test set before evaluation.


In [ ]:
# ── Step 6a: Separate features (X) from the target label (y) ─────────────

# X = the input features — the 24 measurements the model will study
# df[selected_features] picks only our 24 chosen columns
# .values converts the pandas DataFrame into a plain numpy array (faster for ML)
X = df[selected_features].values

# y = the target / answer — what we want the model to predict
# 0 = Benign, 1 = Attack
# .values gives us a plain numpy array of 0s and 1s
y = df["binary_label"].values

print(f"Input features (X): {X.shape}")     # (number of records, number of features)
print(f"Target labels   (y): {y.shape}")    # (number of records,)
print()

# ── Step 6b: Split into training set and test set ─────────────────────────

# train_test_split() randomly splits X and y into four parts:
# X_train, y_train = 80% of data — the model LEARNS from these
# X_test, y_test   = 20% of data — the model is EVALUATED on these (hidden during training)
# test_size=0.20   = 20% goes to test set
# random_state     = our fixed seed so the split is the same every time we run this
# stratify=y       = make sure the Benign/Attack ratio is the same in BOTH splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,          # preserves the class balance in both splits
)

print(f"Training set size: {X_train.shape[0]:,} records (80%)")
print(f"Test set size:     {X_test.shape[0]:,} records (20%)")
print()

# ── Step 6c: Normalise (scale) the numbers ───────────────────────────────

# StandardScaler transforms each feature to have mean=0 and std=1
# This prevents features with large numbers from dominating the model
scaler = StandardScaler()

# .fit_transform(X_train) = learn the mean and std FROM training data, then scale it
# We learn from training data only — the model must not peek at test data
X_train_scaled = scaler.fit_transform(X_train)

# .transform(X_test) = apply the SAME scaling (from training) to the test data
# We do NOT fit again on test data — that would leak information
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print(f"Before scaling — 'Flow Duration' range: {X_train[:, 0].min():.0f} to {X_train[:, 0].max():.0f}")
print(f"After scaling  — 'Flow Duration' range: {X_train_scaled[:, 0].min():.2f} to {X_train_scaled[:, 0].max():.2f}")


## Step 7 — Model 1: Random Forest

### How does a Random Forest work?

Imagine you are trying to decide if a bag is suspicious at an airport.

1. You ask **100 different security guards** to look at the bag
2. Each guard only looks at **some of the clues** (weight, shape, contents, etc.)
3. Each guard makes a decision: "Safe" or "Suspicious"
4. The majority vote wins

That is a Random Forest. Each "guard" is a **Decision Tree** — a chain of yes/no questions. By combining 100 of them (called an "ensemble"), we get a much more reliable result than any single tree.

### Key settings explained:
- `n_estimators=100` → build 100 trees (guards)
- `max_depth=20` → each tree can ask at most 20 questions deep
- `min_samples_split=5` → a node must have at least 5 records before it can split further (prevents overfitting)
- `n_jobs=-1` → use ALL available CPU cores simultaneously (much faster)
- `random_state=RANDOM_STATE` → reproducible tree randomness


In [ ]:
# Tell the user training is starting
print("Training Random Forest model... (this takes about 30-60 seconds)")
print()

# Create the Random Forest model with our chosen settings
# RandomForestClassifier = this is the actual ML algorithm class from scikit-learn
rf_model = RandomForestClassifier(
    n_estimators=100,        # train 100 decision trees
    max_depth=20,            # each tree can go at most 20 levels deep
    min_samples_split=5,     # need at least 5 records to split a tree node
    n_jobs=-1,               # use all CPU cores in parallel (speeds up training)
    random_state=RANDOM_STATE,  # fixed seed for reproducibility
)

# .fit() = the actual training step
# We pass in the SCALED training features and the correct labels
# The model studies these 80,000 records and learns patterns
rf_model.fit(X_train_scaled, y_train)

print("Random Forest training complete!")


In [ ]:
# ── Evaluate the Random Forest ────────────────────────────────────────────

# .predict() = run the trained model on the TEST data (20,000 records it never saw)
# Returns an array of 0s and 1s (predicted labels)
rf_predictions = rf_model.predict(X_test_scaled)

# Calculate each metric by comparing predicted labels vs true labels (y_test)

# Accuracy = (correct predictions) / (total predictions)
# Example: if 19,500 out of 20,000 were correct → accuracy = 97.5%
rf_accuracy = accuracy_score(y_test, rf_predictions)

# Precision = of everything labelled "Attack", how many were ACTUALLY attacks?
# A low precision = lots of false alarms (calling safe traffic "attack")
rf_precision = precision_score(y_test, rf_predictions, zero_division=0)

# Recall = of all REAL attacks, how many did the model CATCH?
# A low recall = lots of missed attacks (calling attacks "safe")
# For security systems, recall is especially important — missing an attack is dangerous
rf_recall = recall_score(y_test, rf_predictions, zero_division=0)

# F1 Score = the harmonic mean of precision and recall
# It is a single balanced score: a high F1 means both precision AND recall are good
# Formula: F1 = 2 * (precision * recall) / (precision + recall)
rf_f1 = f1_score(y_test, rf_predictions, zero_division=0)

# Print the results (multiply by 100 to show as percentages)
print("=== Random Forest Performance ===")
print(f"Accuracy:  {rf_accuracy * 100:.2f}%")
print(f"Precision: {rf_precision * 100:.2f}%")
print(f"Recall:    {rf_recall * 100:.2f}%")
print(f"F1 Score:  {rf_f1 * 100:.2f}%")
print()

# classification_report prints all metrics in a formatted table
# target_names = human-readable names for class 0 and class 1
print("Full Classification Report:")
print(classification_report(y_test, rf_predictions, target_names=["Benign", "Attack"]))


In [ ]:
# ── Confusion Matrix for Random Forest ───────────────────────────────────

# A confusion matrix is a 2x2 grid that shows:
#   True Positives  (TP): Real attacks that we correctly predicted as Attack
#   True Negatives  (TN): Real benign records we correctly predicted as Benign
#   False Positives (FP): Benign traffic we wrongly called Attack (false alarm)
#   False Negatives (FN): Real attacks we missed and called Benign (dangerous!)
#
# Layout:
#            Predicted Benign | Predicted Attack
# Actual Benign |    TN        |      FP
# Actual Attack |    FN        |      TP

# confusion_matrix() computes the 4 numbers from true vs predicted labels
cm_rf = confusion_matrix(y_test, rf_predictions)

# Draw a heatmap of the confusion matrix
plt.figure(figsize=(5, 4))

# sns.heatmap() draws the coloured grid
# annot=True = write the number inside each square
# fmt="d"    = format numbers as integers (not scientific notation)
# cmap="Blues" = use a blue colour gradient
# xticklabels / yticklabels = labels for the columns and rows
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted Benign", "Predicted Attack"],
            yticklabels=["Actual Benign", "Actual Attack"])

plt.title("Random Forest — Confusion Matrix")
# xlabel = label for the horizontal axis (what we predicted)
plt.xlabel("Predicted Label")
# ylabel = label for the vertical axis (what was actually true)
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

# Also print the raw numbers
print("Confusion Matrix (raw numbers):")
print(f"True Negatives  (safe, correctly called safe):   {cm_rf[0,0]:,}")
print(f"False Positives (safe, wrongly called attack):   {cm_rf[0,1]:,}")
print(f"False Negatives (attack, missed as safe):        {cm_rf[1,0]:,}")
print(f"True Positives  (attack, correctly caught):      {cm_rf[1,1]:,}")


In [ ]:
# ── Feature Importances ───────────────────────────────────────────────────

# Random Forest can tell us which features it relied on most during training
# .feature_importances_ is an array of importance scores (0 to 1) for each feature
# Higher score = that feature was more useful for making decisions
importances = rf_model.feature_importances_

# How many features to show in the chart (at most 15, or fewer if we have fewer)
top_n = min(15, len(selected_features))

# np.argsort() returns the INDICES that would sort the array in ascending order
# [::-1] reverses it to descending (most important first)
# [:top_n] takes only the first top_n indices
indices = np.argsort(importances)[::-1][:top_n]

# Draw the bar chart of feature importances
plt.figure(figsize=(10, 5))

# range(top_n) = x positions: 0, 1, 2, ..., top_n-1
# importances[indices] = importance scores in descending order
plt.bar(range(top_n), importances[indices], color="#4a90d9", edgecolor="black", linewidth=0.4)

# Set the x-axis tick labels to the actual feature names
# selected_features[i] = the name of the i-th feature
plt.xticks(range(top_n), [selected_features[i] for i in indices], rotation=45, ha="right", fontsize=9)

plt.ylabel("Importance Score")
plt.title(f"Top {top_n} Most Important Features — Random Forest")
plt.tight_layout()
plt.show()

print("The features with the highest bars are the ones the Random Forest used most to make decisions.")
print("Low-importance features barely affected the outcome at all.")


## Step 8 — Model 2: Neural Network (MLP)

### What is a Neural Network?

A **Multi-Layer Perceptron (MLP)** is a type of artificial neural network inspired loosely by how neurons in the brain connect.

Think of it as a chain of transformations:

```
Input (24 numbers)
     ↓
Layer 1 — 128 neurons — look for simple patterns
     ↓
Layer 2 — 64 neurons  — combine those patterns
     ↓
Layer 3 — 32 neurons  — refine further
     ↓
Output — 1 number between 0 and 1
         (close to 0 = Benign, close to 1 = Attack)
```

### Key terms explained:

| Term | Plain English |
|---|---|
| **Dense layer** | Every input connects to every neuron in this layer |
| **ReLU activation** | If the value is negative, make it 0; otherwise keep it. This adds non-linearity (lets the network learn curves, not just straight lines) |
| **Batch Normalization** | After each layer, rescale the numbers to stay in a reasonable range — makes training more stable |
| **Dropout** | During training, randomly turn off some neurons. Forces the network to not rely on any single neuron — reduces overfitting |
| **Sigmoid activation** | Squashes the final number to be between 0 and 1 (like a probability) |
| **Adam optimizer** | Smart algorithm that adjusts the learning speed automatically |
| **Binary Cross-Entropy loss** | Measures how wrong the prediction is for a yes/no question — the number we try to minimise during training |
| **Backpropagation** | After seeing how wrong it was, the network adjusts all its weights to do better next time |


In [ ]:
# ── Build the Neural Network architecture ────────────────────────────────

# n_features = how many input values the network receives per record
# X_train_scaled.shape[1] = number of columns = number of features
n_features = X_train_scaled.shape[1]

print(f"Building neural network with {n_features} input features...")
print()

# keras.Sequential = a stack of layers processed in order (input → layer1 → layer2 → output)
nn_model = keras.Sequential([

    # Input layer: tells the network how many numbers to expect per record
    # shape=(n_features,) = one record is a list of n_features numbers
    layers.Input(shape=(n_features,)),

    # Dense(128) = a fully-connected layer with 128 neurons
    # activation="relu" = ReLU activation: turns negative values to 0, keeps positive values
    # This is where the network first tries to find patterns in the input
    layers.Dense(128, activation="relu"),

    # BatchNormalization: normalises the outputs of the previous layer
    # Prevents values from becoming too large or too small during training
    # Makes learning faster and more stable
    layers.BatchNormalization(),

    # Dropout(0.30) = randomly turn off 30% of neurons during EACH training step
    # This forces the network to learn redundant paths — not to rely on one neuron
    # Dropout is turned OFF during prediction (it only helps during training)
    layers.Dropout(0.30),

    # Second Dense layer with 64 neurons — learns higher-level combinations of patterns
    layers.Dense(64, activation="relu"),

    # Batch norm again after the second layer
    layers.BatchNormalization(),

    # Dropout 20% — a bit gentler here because the network is narrower
    layers.Dropout(0.20),

    # Third Dense layer with 32 neurons — final refinement before the output
    layers.Dense(32, activation="relu"),

    # Output layer: just 1 neuron
    # activation="sigmoid" = squashes the output to a number between 0 and 1
    # Interpretation: 0.0 = "definitely Benign", 1.0 = "definitely Attack"
    # We will use threshold 0.5: >= 0.5 → Attack, < 0.5 → Benign
    layers.Dense(1, activation="sigmoid"),

], name="NetGuard_MLP")   # give the model a name for display purposes

# Compile = configure how the model will learn
# optimizer="Adam" = use the Adam algorithm to update weights (smart, adaptive learning rate)
# learning_rate=0.001 = how big each step is when adjusting weights (small = careful)
# loss="binary_crossentropy" = the formula we use to measure how wrong predictions are
#   For binary yes/no problems, this is the standard choice
# metrics=["accuracy"] = also track accuracy during training so we can see progress
nn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Print a summary of the architecture — shows each layer, its shape, and parameter count
nn_model.summary()


In [ ]:
# ── Set up training helpers (callbacks) ──────────────────────────────────

# Callbacks are functions that run automatically during training after each epoch

# EarlyStopping: stops training if the model stops improving
# monitor="val_loss" = watch the validation loss (error on data not used for training)
# patience=3 = if val_loss does not improve for 3 consecutive epochs, stop
# restore_best_weights=True = after stopping, go back to the weights from the best epoch
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

# ReduceLROnPlateau: if learning stalls, make the learning rate smaller
# monitor="val_loss" = watch validation loss
# factor=0.5 = multiply the learning rate by 0.5 (halve it) when stalling
# patience=2 = wait 2 epochs before reducing
# min_lr=1e-6 = never go below this learning rate
# verbose=1 = print a message when the rate is reduced
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1,
)

# ── Train the neural network ──────────────────────────────────────────────
print("Training neural network... (may take a few minutes)")
print("Each epoch = one full pass through the training data")
print()

# nn_model.fit() = the actual training
# X_train_scaled, y_train = our training features and labels
# validation_split=0.15 = hold back 15% of training data as a validation set
#   (the model never trains on these but uses them to check for overfitting)
# epochs=25 = train for at most 25 full passes through the data
#   (EarlyStopping may stop it earlier if the model stops improving)
# batch_size=512 = process 512 records at a time before updating weights
#   Larger batches = more stable but slower; 512 is a good balance
# callbacks = our early stopping and learning rate reducer
# verbose=1 = print one line per epoch showing loss and accuracy
history = nn_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=25,
    batch_size=512,
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)

print()
print("Neural network training complete!")


In [ ]:
# ── Plot training history ─────────────────────────────────────────────────

# 'history' stores what happened during training: accuracy and loss per epoch
# We plot these to check if the model learned well and did not overfit

# Create a figure with 2 side-by-side charts
# fig = the whole figure canvas
# (ax1, ax2) = the two individual chart areas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ── Left chart: Accuracy over epochs ─────────────────────────────────────
# history.history["accuracy"] = training accuracy at each epoch
# history.history["val_accuracy"] = validation accuracy at each epoch
# If val_accuracy is much lower than training accuracy → overfitting
ax1.plot(history.history["accuracy"], label="Training Accuracy")
ax1.plot(history.history["val_accuracy"], label="Validation Accuracy", linestyle="--")
ax1.set_title("Neural Network — Accuracy Over Training")
ax1.set_xlabel("Epoch (training pass number)")
ax1.set_ylabel("Accuracy (higher = better)")
ax1.legend()
ax1.grid(alpha=0.3)  # light grid lines for readability

# ── Right chart: Loss over epochs ────────────────────────────────────────
# Loss = how wrong the model's predictions are (lower = better)
# If validation loss goes back up after a point → overfitting started there
ax2.plot(history.history["loss"], label="Training Loss")
ax2.plot(history.history["val_loss"], label="Validation Loss", linestyle="--")
ax2.set_title("Neural Network — Loss Over Training")
ax2.set_xlabel("Epoch (training pass number)")
ax2.set_ylabel("Binary Cross-Entropy Loss (lower = better)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("If both training and validation lines are close together: good (no overfitting)")
print("If they diverge wildly: the model memorised training data and does not generalise")


In [ ]:
# ── Evaluate the Neural Network ───────────────────────────────────────────

# nn_model.predict() = run the trained neural net on test data
# Returns probabilities between 0 and 1 for each record
# .flatten() = convert from 2D array [[0.9], [0.1], ...] to 1D [0.9, 0.1, ...]
# verbose=0 = do not print a progress bar during prediction
nn_probs = nn_model.predict(X_test_scaled, verbose=0).flatten()

# Convert probabilities to 0 or 1 using threshold 0.5
# >= 0.5 → 1 (Attack), < 0.5 → 0 (Benign)
# .astype(int) converts True/False boolean array to 1/0 integers
nn_predictions = (nn_probs >= 0.5).astype(int)

# Calculate all the same metrics as we did for Random Forest
nn_accuracy  = accuracy_score(y_test, nn_predictions)
nn_precision = precision_score(y_test, nn_predictions, zero_division=0)
nn_recall    = recall_score(y_test, nn_predictions, zero_division=0)
nn_f1        = f1_score(y_test, nn_predictions, zero_division=0)

# Print results
print("=== Neural Network (MLP) Performance ===")
print(f"Accuracy:  {nn_accuracy * 100:.2f}%")
print(f"Precision: {nn_precision * 100:.2f}%")
print(f"Recall:    {nn_recall * 100:.2f}%")
print(f"F1 Score:  {nn_f1 * 100:.2f}%")
print()
print("Full Classification Report:")
print(classification_report(y_test, nn_predictions, target_names=["Benign", "Attack"]))

# Draw the confusion matrix for the neural network
cm_nn = confusion_matrix(y_test, nn_predictions)
plt.figure(figsize=(5, 4))
# Orange colormap to visually distinguish it from the Random Forest (blue)
sns.heatmap(cm_nn, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Predicted Benign", "Predicted Attack"],
            yticklabels=["Actual Benign", "Actual Attack"])
plt.title("Neural Network — Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()


## Step 9 — Compare Both Models

Now we put the two models side by side and compare all four metrics.

### Quick recap of what each metric means:

| Metric | Simple explanation |
|---|---|
| **Accuracy** | Out of all 20,000 test records, what % did we get right? |
| **Precision** | When we said "Attack", how often were we actually right? (avoids false alarms) |
| **Recall** | Out of all REAL attacks, how many did we catch? (avoids missed attacks) |
| **F1 Score** | A single number that balances precision and recall — the most useful overall metric |

For a security system, **Recall is the most important metric**. Missing a real attack (False Negative) is more dangerous than a false alarm (False Positive).


In [ ]:
# ── Build a comparison table ──────────────────────────────────────────────

# pd.DataFrame() creates a table from a dictionary
# Each key = column name, each value = list of values for that column
# round(..., 2) = round to 2 decimal places
comparison = pd.DataFrame({
    "Model": ["Random Forest", "Neural Network (MLP)"],
    "Accuracy (%)":  [round(rf_accuracy  * 100, 2), round(nn_accuracy  * 100, 2)],
    "Precision (%)": [round(rf_precision * 100, 2), round(nn_precision * 100, 2)],
    "Recall (%)":    [round(rf_recall    * 100, 2), round(nn_recall    * 100, 2)],
    "F1 Score (%)":  [round(rf_f1        * 100, 2), round(nn_f1        * 100, 2)],
}).set_index("Model")   # use the Model column as the row label

# Print the table
print("Side-by-side comparison:")
print(comparison.to_string())


In [ ]:
# ── Draw the comparison bar chart ─────────────────────────────────────────

# The 4 metrics we will show on the chart
metrics = ["Accuracy (%)", "Precision (%)", "Recall (%)", "F1 Score (%)"]

# x = positions on the x-axis: 0, 1, 2, 3 (one for each metric)
x = np.arange(len(metrics))

# width = how wide each bar is; 0.35 leaves a gap between the two models' bars
width = 0.35

# Create the figure
fig, ax = plt.subplots(figsize=(10, 5))

# Draw Random Forest bars — shifted slightly LEFT of center (x - width/2)
bars1 = ax.bar(x - width/2,
               comparison.loc["Random Forest", metrics],
               width,
               label="Random Forest",
               color="#4a90d9",        # blue
               edgecolor="black",
               linewidth=0.4)

# Draw Neural Network bars — shifted slightly RIGHT of center (x + width/2)
bars2 = ax.bar(x + width/2,
               comparison.loc["Neural Network (MLP)", metrics],
               width,
               label="Neural Network (MLP)",
               color="#e67e22",        # orange
               edgecolor="black",
               linewidth=0.4)

ax.set_ylabel("Score (%)")
ax.set_title("Random Forest vs Neural Network — All Metrics")

# Put metric names on the x-axis
ax.set_xticks(x)
ax.set_xticklabels(metrics)

# Start the y-axis at 80% so small differences are more visible
ax.set_ylim(80, 101)

# Show the legend (which colour is which model)
ax.legend()

# Add the percentage value on top of each bar
# ax.bar_label() places text annotations at the top of each bar
ax.bar_label(bars1, fmt="%.1f%%", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.1f%%", padding=3, fontsize=9)

# Light horizontal grid lines for readability
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


## Step 10 — Interactive Prediction Demo

This is the fun part. We pick a random record from the test set and ask both models to predict whether it is Benign or an Attack.

This is exactly what a real intrusion detection system would do — except in production it would run on live network flows, not stored records.

**What the function does:**
1. Pick one random network connection record
2. Ask Random Forest: "Attack or Benign?" (also gives confidence %)
3. Ask Neural Network: "What is the attack probability?" (0–100%)
4. Compare both answers to the true label


In [ ]:
# ── Define the prediction function ───────────────────────────────────────

def predict_traffic(record_index=None):
    """
    Predict whether a single network flow record is Benign or an Attack.

    Parameters:
        record_index: (optional) which test record to use
                      If None, picks a random record from the test set
    """

    # If no specific record was requested, pick one at random
    # np.random.randint(0, len(X_test)) = random integer between 0 and test set size
    if record_index is None:
        record_index = np.random.randint(0, len(X_test))

    # Get the scaled feature values for this one record
    # X_test_scaled[record_index] = one row (1D array of 24 numbers)
    # .reshape(1, -1) = reshape into 2D: [[...]] because models expect batch input
    record = X_test_scaled[record_index].reshape(1, -1)

    # Look up the TRUE label for this record
    # y_test[record_index] = 0 (Benign) or 1 (Attack)
    actual = "Attack" if y_test[record_index] == 1 else "Benign"

    # ── Random Forest prediction ──────────────────────────────────────────
    # .predict() returns the winning class label (0 or 1)
    rf_pred = rf_model.predict(record)[0]

    # .predict_proba() returns probabilities for each class [P(Benign), P(Attack)]
    # [rf_pred] picks the probability of the predicted class → that is the confidence
    rf_conf = rf_model.predict_proba(record)[0][rf_pred]

    # ── Neural Network prediction ─────────────────────────────────────────
    # .predict() returns the sigmoid output: a probability between 0 and 1
    # verbose=0 = no progress bar printed
    # [0][0] = get the single float value from the [[value]] output
    nn_prob = float(nn_model.predict(record, verbose=0)[0][0])

    # Convert probability to label: >= 0.5 → Attack, else → Benign
    nn_pred = int(nn_prob >= 0.5)

    # Convert 0/1 predictions to human-readable strings
    rf_label = "Attack" if rf_pred == 1 else "Benign"
    nn_label = "Attack" if nn_pred == 1 else "Benign"

    # Print the result for this record
    print(f"Test Record #{record_index}")
    print(f"  True answer:          {actual}")
    print(f"  Random Forest says:   {rf_label}  (confidence: {rf_conf*100:.1f}%)")
    print(f"  Neural Network says:  {nn_label}  (attack probability: {nn_prob*100:.1f}%)")

    # Simple right/wrong indicator
    rf_correct = "✓" if rf_label == actual else "✗"
    nn_correct = "✓" if nn_label == actual else "✗"
    print(f"  RF correct? {rf_correct}   NN correct? {nn_correct}")
    print()


In [ ]:
# ── Run the prediction function on 5 random test records ─────────────────

print("Running predictions on 5 random network flow records from the test set:")
print("=" * 60)

# _ = we do not need the loop variable (just want to repeat 5 times)
for _ in range(5):
    # Call our function with no argument → picks a random record each time
    predict_traffic()

print("=" * 60)
print()
print("If both models are mostly correct: the training worked well.")
print("An occasional mistake is normal — no model is perfect.")
print()
print("To test a specific record, run: predict_traffic(record_index=42)")


## Summary — What Did We Build?

### The complete pipeline

```
Raw network flow data (CIC-IDS2017)
        ↓
Load 70,000 Benign + 30,000 Attack records
        ↓
Clean: fix infinite values, fill missing values
        ↓
Select 24 informative features
        ↓
Split: 80% train / 20% test  (stratified)
        ↓
Scale: StandardScaler → mean 0, std 1
        ↓
   ┌──────────────────────────────────┐
   │ Model 1: Random Forest            │  ← 100 decision trees voting
   │ Model 2: Neural Network (MLP)     │  ← 3 hidden layers + backpropagation
   └──────────────────────────────────┘
        ↓
Evaluate: Accuracy, Precision, Recall, F1, Confusion Matrix
        ↓
Interactive prediction demo
```

---

### Things to say in a presentation or exam

**What is CIC-IDS2017?**
> "It is a research dataset from the Canadian Institute for Cybersecurity. It contains labelled network flow records — both normal traffic and several attack types — recorded in a controlled lab environment in 2017."

**What is a network flow feature?**
> "It is a statistical summary of a network connection. For example, how long it lasted, how many packets were sent, how big those packets were. The model never sees the actual packet contents — only these measurements."

**Why Random Forest?**
> "It is an ensemble method — it trains 100 decision trees on different subsets of the data and uses a majority vote. It is robust, handles non-linear patterns well, and gives us feature importances."

**Why a neural network?**
> "It can learn complex non-linear patterns. We used backpropagation — after each batch, the model checks how wrong it was, computes the gradient of the loss, and adjusts all weights slightly to reduce the error."

**What does Recall mean and why is it important here?**
> "Recall measures what fraction of real attacks we actually detected. In a security system, missing a real attack is more dangerous than a false alarm, so we care about recall more than precision."

**What are the limitations?**
> "The dataset is from 2017. Attack patterns change. This model would need periodic retraining. Also, we used a 100,000-record sample from one day's worth of traffic. Results on live production traffic may differ."
